# 02 – Parametric VaR Baseline

**Goal:** Initialise the portfolio, estimate covariance (rolling + EWMA),
run the 1-minute parametric VaR replay, and plot the results.

**Prerequisite:** Run notebook 01 first to download and save the data.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

import config
from src.data_loader    import load_saved_data
from src.preprocess     import compute_log_returns, save_processed_data
from src.portfolio      import initialize_portfolio
from src.covariance     import rolling_covariance, ewma_covariance, compare_covariance_matrices
from src.parametric_var import run_parametric_var_replay

## 1. Load Data

In [ ]:
daily    = load_saved_data('daily_prices.csv',         config.DATA_RAW)
intraday = load_saved_data('intraday_1min_prices.csv', config.DATA_RAW)
log_returns = compute_log_returns(daily)
save_processed_data(log_returns, 'daily_log_returns.csv')

## 2. Initialise Portfolio

In [ ]:
portfolio = initialize_portfolio(intraday.iloc[0])
shares    = portfolio['shares']
print(shares)

## 3. Estimate Covariance

In [ ]:
cov_roll = rolling_covariance(log_returns)
cov_ewma = ewma_covariance(log_returns)
comp = compare_covariance_matrices(cov_roll, cov_ewma)
comp

## 4. Run Parametric VaR Replay

In [ ]:
results = run_parametric_var_replay(shares, intraday, cov_roll, cov_ewma, save=True)
results.head()

## 5. Visualise Results

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Portfolio value
axes[0].plot(results['portfolio_value'], color='navy')
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].set_title('Portfolio Value — Intraday Replay')

# 99% VaR comparison
axes[1].plot(results['var_99_rolling'], label='Rolling 99% VaR', color='crimson')
axes[1].plot(results['var_99_ewma'],    label='EWMA 99% VaR',    color='darkorange', linestyle='--')
axes[1].set_ylabel('VaR ($)')
axes[1].set_title('99% Parametric VaR: Rolling vs EWMA')
axes[1].legend()

# Latency
axes[2].plot(results['latency_rolling_ms'], label='Rolling', color='steelblue', alpha=0.7)
axes[2].plot(results['latency_ewma_ms'],    label='EWMA',    color='seagreen',  alpha=0.7)
axes[2].set_ylabel('Latency (ms)')
axes[2].set_title('Per-Bar Computation Latency')
axes[2].legend()

plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/var_comparison.png', dpi=150)
plt.show()